# Kafka CDC Processing with Spark Structured Streaming

This notebook is a project-grade reference implementation for consuming Debezium CDC events from Kafka/Redpanda and transforming them into analytics-ready datasets.

## Pipeline
PostgreSQL -> Debezium (Kafka Connect) -> Redpanda/Kafka -> Spark Structured Streaming -> Parquet/Delta sink

## Scope
- topics: `opsdb.public.sales_order`, `opsdb.public.purchase_order`, `opsdb.public.production`
- operations: create/update/delete/snapshot
- sink strategy: portable local parquet (default) and optional Delta

## Runtime Notes (Local vs Databricks)

Use the correct Kafka bootstrap endpoint for your runtime:
- local PySpark on your machine: `localhost:19092`
- container-to-container network: `redpanda:9092`
- Databricks workspace: local Docker endpoints are not reachable directly unless you provide network routing (VPN, tunnel, or public broker endpoint).

If Databricks cannot reach your local broker, run this notebook locally in VS Code/Jupyter with PySpark.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType

KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP", "localhost:19092")
TOPICS = [
    "opsdb.public.sales_order",
    "opsdb.public.purchase_order",
    "opsdb.public.production",
]
CHECKPOINT_BASE = os.getenv("STREAM_CHECKPOINT_BASE", "/tmp/kafka_cdc_checkpoints")
OUTPUT_BASE = os.getenv("STREAM_OUTPUT_BASE", "/tmp/kafka_cdc_output")

spark = SparkSession.builder.appName("KafkaCDCStructuredStreaming").getOrCreate()
print("Spark version:", spark.version)
print("Kafka bootstrap:", KAFKA_BOOTSTRAP)

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:577)
	at org.apache.spark.util.Utils$.$anonfun$getCurrentUserName$1(Utils.scala:2416)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.util.Utils$.getCurrentUserName(Utils.scala:2416)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:334)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
	at java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:483)
	at py4j.reflection.Metho

## Step 1: Read Kafka Stream

The Kafka source returns key/value as bytes plus topic metadata. We keep topic and ingestion timestamp for lineage.

In [ ]:
df_kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", ",".join(TOPICS))
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

df_events = df_kafka_raw.select(
    col("topic").alias("source_topic"),
    col("timestamp").alias("kafka_ingest_ts"),
    col("value").cast("string").alias("event_payload")
)

df_events.printSchema()

## Step 2: Parse Debezium Event Envelope

Depending on converter settings, Debezium can emit either:
- flat envelope (`op`, `before`, `after`, `ts_ms` at root), or
- wrapped envelope (`payload.op`, `payload.before`, ...).

The parser below supports both forms.

In [ ]:
payload_schema = StructType([
    StructField("op", StringType(), True),
    StructField("before", StringType(), True),
    StructField("after", StringType(), True),
    StructField("ts_ms", LongType(), True),
])

debezium_schema = StructType([
    StructField("op", StringType(), True),
    StructField("before", StringType(), True),
    StructField("after", StringType(), True),
    StructField("ts_ms", LongType(), True),
    StructField("payload", payload_schema, True),
])

parsed = df_events.select(
    col("source_topic"),
    col("kafka_ingest_ts"),
    from_json(col("event_payload"), debezium_schema).alias("evt")
)

df_cdc = parsed.select(
    col("source_topic"),
    col("kafka_ingest_ts"),
    coalesce(col("evt.op"), col("evt.payload.op")).alias("op"),
    coalesce(col("evt.before"), col("evt.payload.before")).alias("before_json"),
    coalesce(col("evt.after"), col("evt.payload.after")).alias("after_json"),
    to_timestamp((coalesce(col("evt.ts_ms"), col("evt.payload.ts_ms")) / 1000).cast("double")).alias("event_ts")
)

## Step 3: Table-Specific Typed DataFrames

Each topic is parsed into a typed dataframe to keep downstream contracts explicit.

In [ ]:
sales_order_schema = StructType([
    StructField("id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("shipping_status", StringType(), True),
    StructField("order_date", StringType(), True),
])

purchase_order_schema = StructType([
    StructField("id", StringType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("order_date", StringType(), True),
])

production_schema = StructType([
    StructField("id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("planned_date", StringType(), True),
])

df_sales = (
    df_cdc.filter(col("source_topic") == "opsdb.public.sales_order")
    .select(
        col("source_topic"),
        col("kafka_ingest_ts"),
        col("event_ts"),
        col("op"),
        from_json(col("after_json"), sales_order_schema).alias("row")
    )
    .select(
        col("source_topic"),
        col("kafka_ingest_ts"),
        col("event_ts"),
        col("op"),
        col("row.id").alias("order_id"),
        col("row.customer_id"),
        col("row.shipping_status"),
        to_timestamp(col("row.order_date")).alias("order_date")
    )
)

df_purchase = (
    df_cdc.filter(col("source_topic") == "opsdb.public.purchase_order")
    .select(
        col("source_topic"),
        col("kafka_ingest_ts"),
        col("event_ts"),
        col("op"),
        from_json(col("after_json"), purchase_order_schema).alias("row")
    )
    .select(
        col("source_topic"),
        col("kafka_ingest_ts"),
        col("event_ts"),
        col("op"),
        col("row.id").alias("purchase_order_id"),
        col("row.supplier_id"),
        col("row.status"),
        to_timestamp(col("row.order_date")).alias("order_date")
    )
)

df_production = (
    df_cdc.filter(col("source_topic") == "opsdb.public.production")
    .select(
        col("source_topic"),
        col("kafka_ingest_ts"),
        col("event_ts"),
        col("op"),
        from_json(col("after_json"), production_schema).alias("row")
    )
    .select(
        col("source_topic"),
        col("kafka_ingest_ts"),
        col("event_ts"),
        col("op"),
        col("row.id").alias("production_id"),
        col("row.product_id"),
        col("row.status"),
        to_timestamp(col("row.planned_date")).alias("planned_date")
    )
)

## Step 4: Start Streaming Sinks (Portable Local Parquet)

Parquet sink is chosen by default to keep this notebook runnable in local Spark and Databricks environments without additional Delta dependencies.

In [ ]:
query_sales = (
    df_sales.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", f"{OUTPUT_BASE}/sales_order")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/sales_order")
    .trigger(processingTime="10 seconds")
    .start()
)

query_purchase = (
    df_purchase.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", f"{OUTPUT_BASE}/purchase_order")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/purchase_order")
    .trigger(processingTime="10 seconds")
    .start()
)

query_production = (
    df_production.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", f"{OUTPUT_BASE}/production")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/production")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Streaming queries started")
print("Output base:", OUTPUT_BASE)
print("Checkpoint base:", CHECKPOINT_BASE)

## Step 5: Observe Runtime Health

Use this cell while generating events from Shop-App, Company-App, or the simulator.

In [ ]:
for q in spark.streams.active:
    print("Name:", q.name)
    print("ID:", q.id)
    print("Status:", q.status)
    if q.recentProgress:
        print("Last progress:", q.recentProgress[-1])
    print("-" * 80)

## Step 6: Read Curated Output (Batch Analytics)

Once files are produced, load them as static DataFrames and run exploratory checks.

In [ ]:
sales_batch = spark.read.parquet(f"{OUTPUT_BASE}/sales_order")
purchase_batch = spark.read.parquet(f"{OUTPUT_BASE}/purchase_order")
production_batch = spark.read.parquet(f"{OUTPUT_BASE}/production")

sales_batch.groupBy("shipping_status").count().orderBy(col("count").desc()).show(truncate=False)
purchase_batch.groupBy("status").count().orderBy(col("count").desc()).show(truncate=False)
production_batch.groupBy("status").count().orderBy(col("count").desc()).show(truncate=False)

## Step 7: Shutdown

Stop all active streams at the end of the session to release resources.

In [ ]:
for q in spark.streams.active:
    q.stop()

print("All active streams have been stopped")